In [ ]:
import os
import re
import time
from datetime import datetime
from pathlib import Path

import pandas as pd
import requests

GITHUB_TOKEN = os.getenv("GITHUB_TOKEN", "")
TARGET_NEW_URLS = 100
CUTOFF = "2022-11-01T00:00:00Z"
SEARCH_URL = "https://api.github.com/search/repositories"
OUTPUT_DIR = Path(os.getenv("OUTPUT_DIR", "."))

EXCLUDED_URLS = {
    "https://github.com/MrAnas/ICS324_GROUP_2",
    "https://github.com/KonstantinFilin/ksutils",
    "https://github.com/djbamba/Wildcat-Chat",
    "https://github.com/HassanAlgoz/chefs-website",
    "https://github.com/norahalkhunifer/web",
    "https://github.com/lameessu/Bohemia-website",
    "https://github.com/raisking/ksu_webproject",
    "https://github.com/ksuwannop/ksuwannop.github.io",
    "https://github.com/MAD9013/responsive-website-project1-kaus0025",
    "https://github.com/baileythompson/KSU-IT-Research_Website",
    "https://github.com/ksudhanshu348/ksudhanshu348.github.io",
    "https://github.com/hannahnoelle797/IntroToWebDevSite",
    "https://github.com/hannahnoelle797/AdvWebDevSite",
    "https://github.com/rishamitha/ksu-cafeteria",
    "https://github.com/kaushiki7/kaushiki7.github.io",
    "https://github.com/Narwher/la-hair-atl",
    "https://github.com/pkommineni94/IT6203",
    "https://github.com/surafel911/pizza-shop-webservice",
    "https://github.com/jsweitz1/KSU-Student",
    "https://github.com/othmanKisha/In-Door-Face-Mask-Inspector",
    "https://github.com/ksulem/ksulem.github.io",
    "https://github.com/MujtabaMohsin/Student-Exam-Preparation",
    "https://github.com/Reemaad/KSU-Templates",
    "https://github.com/Kaushikodey/Kaushikodey.github.io",
    "https://github.com/ksuni338/ksuni338.github.io",
    "https://github.com/baileyknez/CapstoneKSU2021",
    "https://github.com/iamanilantony/KSU",
    "https://github.com/ericbutler555/carit",
    "https://github.com/traviskaufman/TravisKaufman_FlashMP3Player",
    "https://github.com/danielgur/ksu-buzz-out",
    "https://github.com/ibalde/colesAppWebPage",
    "https://github.com/jsturzio/websiteacm",
    "https://github.com/lemonad/ksu-online",
    "https://github.com/ksusanth/ksusanth.github.io",
    "https://github.com/santoslab/ksu-cis-501-web",
    "https://github.com/ksuchow99/ksuchow99.github.io",
    "https://github.com/willbreitkreutz/ksu_geog_alumni",
    "https://github.com/hyttjo/KaupunkiUutiset",
    "https://github.com/MaximTikhonov/WebDesignKsu",
    "https://github.com/KeralaStartupMission/ksum_fellows",
    "https://github.com/ksw2342/kau_webstudio_12",
    "https://github.com/dp3/ksudrupal7",
    "https://github.com/zeuxisoo/web-kauppaa",
    "https://github.com/Front-End-Technology/KaushikPizzaWeb",
    "https://github.com/Supan13/kaur-supan-webdev",
    "https://github.com/juhapekkaeloranta/kaupunkifillarit-web",
    "https://github.com/ajaykauthale1/kauthale-ajay-webdev",
    "https://github.com/kaushikveluru/veluru-kaushik-webdev",
    "https://github.com/Supan13/kaur-supan13-webdev",
    "https://github.com/kevin-su/ksu-reactjs",
}
EXCLUDED_URLS = {url.rstrip("/") for url in EXCLUDED_URLS}
EXCLUDED_FULL_NAMES = {
    url.replace("https://github.com/", "").rstrip("/").lower() for url in EXCLUDED_URLS
}

UNIVERSITY_QUERIES = [
    "KFUPM",
    '"King Fahd University of Petroleum and Minerals"',
    '"جامعة الملك فهد للبترول والمعادن"',
    "KSU",
    '"King Saud University"',
    '"جامعة الملك سعود"',
    "KAU",
    '"King Abdulaziz University"',
    '"جامعة الملك عبدالعزيز"',
    '"Saudi student"',
    '"طالب سعودي"',
    '"طالبة سعودية"',
    '"Saudi university"',
    '"جامعة سعودية"',
    '"University of Jeddah"',
    '"جامعة جدة"',
    '"Taif University"',
    '"جامعة الطائف"',
    '"Taibah University"',
    '"جامعة طيبة"',
]

TERM_GROUPS = [
    '(project OR "student project" OR "course project" OR capstone OR "graduation project" OR assignment)',
    "(programming OR coding OR software OR app OR system OR repository)",
    '("machine learning" OR AI OR database OR algorithms OR "data structures" OR cybersecurity)',
]

GOOD_LANGS = {
    "Python",
    "Java",
    "C++",
    "C",
    "C#",
    "JavaScript",
    "TypeScript",
    "PHP",
    "Go",
    "Rust",
    "Kotlin",
    "Swift",
    "MATLAB",
    "R",
    "Jupyter Notebook",
    "HTML",
    "CSS",
    "Shell",
    "Dart",
    "Scala",
    "Objective-C",
}

BAD_HINTS = [
    "awesome",
    "roadmap",
    "notes",
    "cheatsheet",
    "cheat-sheet",
    "curriculum",
    "tutorial",
    "book",
    "slides",
    "lecture",
]

cutoff_dt = datetime.fromisoformat(CUTOFF.replace("Z", "+00:00"))

session = requests.Session()
headers = {
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
}
if GITHUB_TOKEN.strip():
    headers["Authorization"] = f"Bearer {GITHUB_TOKEN.strip()}"
session.headers.update(headers)


def parse_dt(value):
    try:
        return datetime.fromisoformat(value.replace("Z", "+00:00"))
    except (AttributeError, ValueError):
        return None


def normalize_url(url):
    if not url:
        return ""
    url = url.strip()
    if url.endswith(".git"):
        url = url[:-4]
    return url.rstrip("/")


def sleep_if_rate_limited(response):
    if response.status_code == 403:
        remaining = response.headers.get("X-RateLimit-Remaining")
        reset = response.headers.get("X-RateLimit-Reset")
        if remaining == "0" and reset:
            wait_seconds = max(int(reset) - int(time.time()) + 2, 2)
            print(f"Rate limit reached; sleeping for {wait_seconds} seconds.")
            time.sleep(wait_seconds)
            return True
    return False


def github_search(query, page=1, per_page=100):
    params = {
        "q": query,
        "per_page": per_page,
        "page": page,
        "sort": "updated",
        "order": "asc",
    }
    response = session.get(SEARCH_URL, params=params, timeout=60)
    if sleep_if_rate_limited(response):
        response = session.get(SEARCH_URL, params=params, timeout=60)
    response.raise_for_status()
    return response.json()


def text_blob(repo):
    parts = [
        repo.get("name") or "",
        repo.get("full_name") or "",
        repo.get("description") or "",
        " ".join(repo.get("topics") or []),
        repo.get("language") or "",
        repo.get("homepage") or "",
    ]
    return " ".join(parts).lower()


UNI_PATTERN = re.compile(
    r"(kfupm|king fahd university|جامعة الملك فهد|ksu|king saud university|جامعة الملك سعود|kau|king abdulaziz university|جامعة الملك عبدالعزيز|saudi student|طالب سعودي|طالبة سعودية|saudi university|جامعة سعودية|university of jeddah|جامعة جدة|taif university|جامعة الطائف|taibah university|جامعة طيبة)",
    re.I,
)

CODE_PATTERN = re.compile(
    r"\b(project|student|course|capstone|graduation|assignment|programming|coding|software|app|system|repository|machine learning|ai|database|algorithms|data structures|cybersecurity|python|java|c\+\+|c#|javascript|typescript|php|go|rust|kotlin|swift|matlab)\b",
    re.I,
)


def score_repo(repo):
    blob = text_blob(repo)
    score = 0

    if UNI_PATTERN.search(blob):
        score += 6
    if CODE_PATTERN.search(blob):
        score += 5
    if repo.get("language") in GOOD_LANGS:
        score += 4
    if not repo.get("fork", False):
        score += 1
    if not repo.get("archived", False):
        score += 1
    if repo.get("size", 0) > 0:
        score += 1
    if repo.get("stargazers_count", 0) >= 1:
        score += 1
    if any(hint in blob for hint in BAD_HINTS):
        score -= 5

    return score


def is_valid_repo(repo):
    html_url = normalize_url(repo.get("html_url", ""))
    full_name = (repo.get("full_name") or "").lower()
    pushed_at = parse_dt(repo.get("pushed_at", ""))

    if not html_url or not full_name:
        return False
    if html_url in EXCLUDED_URLS or full_name in EXCLUDED_FULL_NAMES:
        return False
    if pushed_at is None or pushed_at >= cutoff_dt:
        return False
    if repo.get("fork", False) or repo.get("archived", False):
        return False
    if repo.get("private", False) or repo.get("disabled", False):
        return False
    if repo.get("size", 0) <= 0:
        return False

    blob = text_blob(repo)
    if not UNI_PATTERN.search(blob):
        return False
    if not CODE_PATTERN.search(blob) and repo.get("language") not in GOOD_LANGS:
        return False
    if any(hint in blob for hint in BAD_HINTS):
        return False

    return True


queries = [
    f"{university} {group} pushed:<2022-11-01 fork:false archived:false"
    for university in UNIVERSITY_QUERIES
    for group in TERM_GROUPS
]
queries.extend(
    [
        '"Saudi student" programming pushed:<2022-11-01 fork:false archived:false',
        '"Saudi student" project pushed:<2022-11-01 fork:false archived:false',
        '"طالب سعودي" برمجة pushed:<2022-11-01 fork:false archived:false',
        '"طالب سعودي" مشروع pushed:<2022-11-01 fork:false archived:false',
        '"جامعة سعودية" programming pushed:<2022-11-01 fork:false archived:false',
        '"جامعة سعودية" project pushed:<2022-11-01 fork:false archived:false',
    ]
)
queries = list(dict.fromkeys(queries))

print(f"Queries: {len(queries)}")

repos_by_id = {}
checked_queries = 0

# Early queries receive a second page to trade API use for coverage.
for index, query in enumerate(queries, start=1):
    if len(repos_by_id) >= TARGET_NEW_URLS * 2:
        break

    checked_queries += 1
    pages = 2 if index <= 15 else 1

    for page in range(1, pages + 1):
        try:
            data = github_search(query, page=page, per_page=100)
            items = data.get("items", [])
        except Exception as error:
            print(
                f"Skipping query page after error: {error}\nQuery: {query}\nPage: {page}\n"
            )
            break

        if not items:
            break

        for repo in items:
            repo_id = repo["id"]
            if repo_id not in repos_by_id and is_valid_repo(repo):
                repos_by_id[repo_id] = repo

        time.sleep(0.15)

    if checked_queries % 10 == 0:
        print(
            f"Checked {checked_queries} queries; collected {len(repos_by_id)} candidates."
        )

print(f"Checked {checked_queries} queries total.")
print(f"Collected {len(repos_by_id)} candidate repositories before ranking.")

if not repos_by_id:
    raise RuntimeError(
        "No repositories found. Set GITHUB_TOKEN and rerun if API limits prevent collection."
    )

rows = [
    {
        "score": score_repo(repo),
        "full_name": repo.get("full_name"),
        "html_url": normalize_url(repo.get("html_url")),
        "description": repo.get("description"),
        "language": repo.get("language"),
        "created_at": repo.get("created_at"),
        "updated_at": repo.get("updated_at"),
        "pushed_at": repo.get("pushed_at"),
        "stargazers_count": repo.get("stargazers_count"),
        "forks_count": repo.get("forks_count"),
        "topics": ", ".join(repo.get("topics") or []),
        "size": repo.get("size"),
    }
    for repo in repos_by_id.values()
]

df = pd.DataFrame(rows)
if df.empty:
    raise RuntimeError("No valid repositories remained after filtering.")

df["pushed_at_dt"] = pd.to_datetime(df["pushed_at"], utc=True, errors="coerce")
df = df.sort_values(
    by=["score", "size", "stargazers_count", "pushed_at_dt"],
    ascending=[False, False, False, True],
).reset_index(drop=True)
df_top = df.head(TARGET_NEW_URLS).copy()

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
csv_path = OUTPUT_DIR / "saudi_university_coding_projects_extra_before_nov2022.csv"
txt_path = OUTPUT_DIR / "saudi_university_coding_projects_extra_before_nov2022.txt"

df_top.drop(columns=["pushed_at_dt"]).to_csv(
    csv_path, index=False, encoding="utf-8-sig"
)
with txt_path.open("w", encoding="utf-8") as output_file:
    output_file.write("\n".join(df_top["html_url"]) + "\n")

print(f"Saved CSV: {csv_path}")
print(f"Saved TXT: {txt_path}")
print(f"Prepared {len(df_top)} non-duplicate repository URLs.")

for index, url in enumerate(df_top["html_url"], start=1):
    print(f"{index:03d}. {url}")

preview = df_top.drop(columns=["pushed_at_dt"], errors="ignore").head(100)
try:
    display(preview)
except NameError:
    print(preview.to_string(index=False))

In [ ]:
import csv
import hashlib
import json
import os
import re
import shutil
import subprocess
import tempfile
from collections import Counter
from pathlib import Path
from urllib.parse import urlparse

# Configure this location for the private research workspace.
DATA_ROOT = Path(
    os.environ.get("CODE_SAMPLE_DATA_ROOT", "./research_data")
).expanduser()
SOURCE_MODE = "hardcoded"  # "hardcoded" or "pdf"
PDF_WITH_URLS = DATA_ROOT / "repository_urls.pdf"

OUT_ROOT = DATA_ROOT
FINAL_ROOT = OUT_ROOT / "trimmed_for_detection"
REPORT_ROOT = OUT_ROOT / "candidate_reports"
SUMMARY_CSV = OUT_ROOT / "summary.csv"
FILE_LEVEL_CSV = OUT_ROOT / "selected_files.csv"
FAILURE_LOG = OUT_ROOT / "failures.txt"

TARGET_FINAL_FILES = 50
TARGET_MIN_LINES = 800
TARGET_MIN_WORDS = 3000
WRITE_SMALL_OUTPUTS = True

MIN_LINES_PER_FILE = 12
MAX_LINES_PER_FILE = 4000
MAX_FILE_BYTES = 1_200_000
MAX_CANDIDATE_FILES_PER_REPO = 60
MAX_SELECTED_FILES_PER_REPO = 25
MARK_LINE = "############"

HARDCODED_URLS = [
    "https://github.com/AbdulkareemKR/Library-system",
    "https://github.com/usernane/ICS_324_Project",
    "https://github.com/Alnasser0/COE451-Project-Secure-ChatApp",
    "https://github.com/d-smith/ds-ksu-mse",
    "https://github.com/FarisHijazi/Secure-FileTransfer",
    "https://github.com/ideodiode/KSU-Ticket-MGMT-System",
    "https://github.com/mohamedFalah/kfupm_Space",
    "https://github.com/ajmalali/auto-register-chrome",
    "https://github.com/m4l3k5/MyPaintShop",
    "https://github.com/WestOnfOrD/JavaSimulatedOS",
    "https://github.com/chrono101/ksu-drupal6-theme",
    "https://github.com/Mokdy/KFUPM-Room-Finder",
    "https://github.com/hadysata/KFUPM-Hidden-Terms",
    "https://github.com/HashemShullar/Intelligent-Control",
    "https://github.com/eligundry/ksu-cs1",
    "https://github.com/eligundry/ksu-cs2",
    "https://github.com/AlexMathew/ksupdates",
    "https://github.com/arawn/ksug-2012-spring-31-summary",
    "https://github.com/hjamaan/AhmedAlshaaby_MscThesis",
    "https://github.com/ammar-faifi/ICS233",
    "https://github.com/hadysata/AutoRegister-KFUPM",
    "https://github.com/spsuaneupane/ksu-capstone-project-app",
    "https://github.com/zbaker94/GameDesign_spring15_FinalProject",
    "https://github.com/FarisHijazi/ICS-485-Machine-Learning",
    "https://github.com/tella10/Kbs-KFUPM",
    "https://github.com/omar-omair/kfupmScheduler",
    "https://github.com/teynon/ksucapstone",
    "https://github.com/dkellycollins/KSU_CIS526",
    "https://github.com/brebory/ksu-rem",
    "https://github.com/aneupane/ksu-capstone-ui",
    "https://github.com/TeamHTTP418/cis726assign5",
    "https://github.com/manishsatasiya/ksu_hrm_school",
    "https://github.com/MohammedFadin/Nitaj.com",
    "https://github.com/jdister1/KSUAdvising",
    "https://github.com/ruting1987/ksun",
    "https://github.com/ibrahimaljalal/KFUPM-AE-581-Project",
    "https://github.com/novayadikomang/ksu",
    "https://github.com/kfupmsapms/sapms",
    "https://github.com/SDILogin/KSU_project413",
    "https://github.com/ibrahimaljalal/KFUPM-COE-510-ROS-Robot-Operating-Systems-Project",
    "https://github.com/semo97/Multipule_Robot_ROS_Drone",
    "https://github.com/rgodbee/CS3550",
    "https://github.com/kuhaha/ksuwiki",
    "https://github.com/danielgur/ksu-buzz-out-backend",
    "https://github.com/ksustudy/ksustudy",
    "https://github.com/paul-hudlow/ksu-6623-project",
    "https://github.com/aratabekov/CampusNavApp",
    "https://github.com/bsteg17/freestuff_finder_ksu",
    "https://github.com/ajmalali/kfupm-maintenance-system",
    "https://github.com/m87i/ICS201",
    "https://github.com/ahankuo/ksu-moodle-app",
    "https://github.com/SalmanYG/ics-department-website",
    "https://github.com/fcm2009/KFUPMTrade",
    "https://github.com/oafsalem/ICS202",
    "https://github.com/ibrahimaljalal/KFUPM-ICS-520-Project-AI",
    "https://github.com/mohamedFalah/KFUPMSocialSpace",
    "https://github.com/2016-ksu-cis-open-house/tree-display",
    "https://github.com/rvpatel92/KSU-Portal",
    "https://github.com/bkowalik/ksug-pres",
    "https://github.com/BrandonCookeDev/KSUeSportsBot",
    "https://github.com/Buzz5aw/ksuwdc.github.io",
    "https://github.com/nodis6831/ksutton",
    "https://github.com/SalmanYG/diet-tracker",
    "https://github.com/dkellycollins/KSU_CIS553",
    "https://github.com/KSUFootball2015/KSUFootball2015.github.io",
    "https://github.com/oafsalem/KFUPMLIBSYS",
    "https://github.com/OmarWKH/162-381",
    "https://github.com/SalmanYG/SWE363-HW1",
    "https://github.com/willbreitkreutz/alumni-map",
    "https://github.com/ruzz311/ksu-countdown",
    "https://github.com/Ali-Asiri-X/ICS-104-project",
    "https://github.com/KFUPMArabi/kfupmarabi.github.io",
    "https://github.com/zahraniana/LibraryMobile",
    "https://github.com/Smast3r/kfupm-webdev-comp",
    "https://github.com/dannydes/ksu-so-scrapper",
    "https://github.com/bandereidm/KFUPMSOC",
    "https://github.com/A7madHD/KFUPM-Maintainace-System",
    "https://github.com/dkellycollins/KSU_CIS560",
    "https://github.com/HashemShullar/SeamCarving",
    "https://github.com/yuta39/KadaiKsuse2_0705",
    "https://github.com/ibrahimaljalal/KFUPM-ICS-520-HW3",
    "https://github.com/joeykiwi/KSUApplication",
    "https://github.com/OmarWKH/324-gamehub",
    "https://github.com/eligundry/speed-programming",
    "https://github.com/fawaz-alesayi/kfupm-clubs",
    "https://github.com/rgodbee/CS3540",
    "https://github.com/hassain-saeed28/KFUPM-parking-management-system",
    "https://github.com/Abdulrahman-Khayat/KFUPM",
    "https://github.com/ZombSmile/Pavlov_Ksu",
    "https://github.com/alaa13212/kfupmOria",
    "https://github.com/alfred316/KSU-CS2",
    "https://github.com/hbahrami/kent",
    "https://github.com/tma02/ksu-hspc-2014-solutions",
    "https://github.com/sandman889/KSUfoodMap",
    "https://github.com/yuta39/KadaiKsuse0705",
    "https://github.com/cpiggott/python-class",
    "https://github.com/yarob55/kfupmsc-android",
    "https://github.com/hacksu/ksu-flash-info",
    "https://github.com/alfred316/hacKSU",
    "https://github.com/rgodbee/CS4150",
]

GITHUB_URL_RE = re.compile(r'https?://[^\s)\]"\'>]+', re.IGNORECASE)

CODE_EXTS = {
    ".py",
    ".js",
    ".ts",
    ".jsx",
    ".tsx",
    ".java",
    ".kt",
    ".kts",
    ".c",
    ".cc",
    ".cpp",
    ".cxx",
    ".h",
    ".hh",
    ".hpp",
    ".cs",
    ".go",
    ".rs",
    ".rb",
    ".php",
    ".swift",
    ".scala",
    ".sql",
    ".html",
    ".css",
    ".scss",
    ".sass",
    ".m",
    ".r",
    ".jl",
    ".sh",
    ".bash",
    ".zsh",
    ".vue",
    ".svelte",
    ".dart",
    ".lua",
    ".pl",
    ".ipynb",
}

BINARY_EXTS = {
    ".png",
    ".jpg",
    ".jpeg",
    ".gif",
    ".bmp",
    ".webp",
    ".ico",
    ".pdf",
    ".zip",
    ".rar",
    ".7z",
    ".tar",
    ".gz",
    ".mp3",
    ".wav",
    ".mp4",
    ".avi",
    ".mov",
    ".mkv",
    ".class",
    ".jar",
    ".exe",
    ".dll",
    ".so",
    ".o",
    ".a",
    ".pyc",
    ".pyo",
    ".woff",
    ".woff2",
    ".ttf",
    ".eot",
    ".map",
}

SKIP_DIRS = {
    ".git",
    ".github",
    ".idea",
    ".vscode",
    "__pycache__",
    "node_modules",
    "vendor",
    "dist",
    "build",
    "out",
    "coverage",
    ".next",
    ".nuxt",
    ".svelte-kit",
    ".cache",
    ".gradle",
    ".mypy_cache",
    ".pytest_cache",
    ".tox",
    "venv",
    ".venv",
    "env",
    "Pods",
    "target",
    "bin",
    "obj",
    "static",
    "assets",
    "images",
    "img",
    "fonts",
}

SKIP_FILES = {
    "package-lock.json",
    "yarn.lock",
    "pnpm-lock.yaml",
    "composer.lock",
    "poetry.lock",
    "Cargo.lock",
    "Gemfile.lock",
    "Pipfile.lock",
    "pubspec.lock",
    "LICENSE",
    "LICENSE.md",
    "COPYING",
    "favicon.ico",
}

GENERATED_MARKERS = [
    "do not edit",
    "this file was generated",
    "generated by",
    "autogenerated",
    "auto-generated",
    "automatically generated",
    "create-react-app",
    "vite",
    "webpack bootstrap",
    "@generated",
]

EXT_BASE_BONUS = {
    ".py": 90,
    ".java": 88,
    ".kt": 88,
    ".kts": 70,
    ".c": 85,
    ".cc": 85,
    ".cpp": 85,
    ".cxx": 85,
    ".h": 55,
    ".hh": 55,
    ".hpp": 55,
    ".cs": 90,
    ".go": 90,
    ".rs": 92,
    ".rb": 80,
    ".php": 80,
    ".swift": 84,
    ".scala": 82,
    ".sql": 82,
    ".js": 78,
    ".ts": 82,
    ".jsx": 64,
    ".tsx": 68,
    ".vue": 70,
    ".svelte": 70,
    ".dart": 78,
    ".m": 74,
    ".r": 74,
    ".jl": 74,
    ".sh": 56,
    ".bash": 56,
    ".zsh": 56,
    ".html": 28,
    ".css": 14,
    ".scss": 18,
    ".sass": 18,
    ".lua": 72,
    ".pl": 70,
    ".ipynb": 78,
}

BONUS_PATH_PATTERNS = [
    (
        r"(^|/)(src|app|server|backend|api|core|lib|internal|pkg|cmd|domain|logic|engine)(/|$)",
        35,
    ),
    (
        r"(^|/)(controllers?|services?|handlers?|routes?|routers?|middleware|guards?|auth|security)(/|$)",
        45,
    ),
    (
        r"(^|/)(models?|entities?|dto|schemas?|database|db|repositories?|dao|migrations?)(/|$)",
        35,
    ),
    (
        r"(^|/)(parser|compiler|solver|algorithm|algorithms|graph|tree|heap|search|sort)(/|$)",
        35,
    ),
    (r"(^|/)(utils?|helpers?|shared|common)(/|$)", 18),
]

PENALTY_PATH_PATTERNS = [
    (
        r"(^|/)(test|tests|testing|__tests__|spec|specs|fixtures|mocks|mock|sample|samples|example|examples)(/|$)",
        -35,
    ),
    (
        r"(^|/)(public|docs?|documentation|assets?|images?|img|fonts?|styles?|css)(/|$)",
        -28,
    ),
    (
        r"(^|/)(vendor|third_party|third-party|external|generated|dist|build|coverage)(/|$)",
        -45,
    ),
]


def run(command, cwd=None):
    process = subprocess.run(
        command, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    return process.returncode, process.stdout


def normalize_url(url):
    return (url or "").strip().rstrip(".,;)")


def ordered_unique(items):
    seen = set()
    return [item for item in items if not (item in seen or seen.add(item))]


def is_github_url(url):
    try:
        return "github.com" in urlparse(url).netloc.lower()
    except ValueError:
        return False


def read_urls_from_pdf(pdf_path):
    if not pdf_path.exists():
        raise FileNotFoundError(f"PDF not found: {pdf_path}")

    try:
        from pypdf import PdfReader
    except ImportError as exc:
        raise ImportError(
            "PDF input requires pypdf. Install it with: pip install pypdf"
        ) from exc

    urls = []
    for page in PdfReader(str(pdf_path)).pages:
        for raw_url in GITHUB_URL_RE.findall(page.extract_text() or ""):
            url = normalize_url(raw_url)
            if is_github_url(url):
                urls.append(url)
    return ordered_unique(urls)


def load_urls():
    urls = (
        read_urls_from_pdf(PDF_WITH_URLS)
        if SOURCE_MODE.lower() == "pdf"
        else HARDCODED_URLS
    )
    urls = [
        url for url in ordered_unique(map(normalize_url, urls)) if is_github_url(url)
    ]
    if not urls:
        raise ValueError("No GitHub URLs were loaded.")
    return urls


def gh_parse(url):
    match = re.match(r"https?://github\.com/([^/]+)/([^/]+)(?:/(.*))?$", url.strip())
    if not match:
        return None

    owner, repo, tail = (
        match.group(1),
        match.group(2).removesuffix(".git"),
        (match.group(3) or "").strip("/"),
    )
    info = {
        "owner": owner,
        "repo": repo,
        "kind": "repo",
        "subpath": None,
        "raw_url": url,
    }

    if tail.startswith("tree/") or tail.startswith("blob/"):
        parts = tail.split("/", 2)
        info["kind"] = parts[0]
        info["subpath"] = parts[2] if len(parts) > 2 else None
    return info


def safe_slug(text):
    return re.sub(r"[^A-Za-z0-9._-]+", "_", text).strip("._") or "item"


def output_stem(info, url):
    digest = hashlib.sha1(url.encode("utf-8")).hexdigest()[:10]
    return safe_slug(f"{info['owner']}__{info['repo']}__{digest}")


def git_clone_repo(owner, repo, destination):
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.rmtree(destination, ignore_errors=True)

    command = [
        "git",
        "clone",
        "--depth",
        "1",
        "--single-branch",
        f"https://github.com/{owner}/{repo}.git",
        str(destination),
    ]
    return_code, output = run(command)
    if return_code == 0:
        return True, output

    shutil.rmtree(destination, ignore_errors=True)
    retry_command = [
        "git",
        "clone",
        "--depth",
        "1",
        f"https://github.com/{owner}/{repo}.git",
        str(destination),
    ]
    return_code, retry_output = run(retry_command)
    return return_code == 0, f"{output}\n\nRETRY:\n{retry_output}"


def is_probably_binary(path):
    try:
        chunk = path.read_bytes()[:8192]
    except OSError:
        return True
    if b"\x00" in chunk:
        return True
    if not chunk:
        return False
    non_text = sum(byte < 9 or 13 < byte < 32 for byte in chunk)
    return non_text / len(chunk) > 0.20


def read_text_file(path):
    for encoding in ("utf-8", "utf-8-sig", "latin-1", "cp1252"):
        try:
            return path.read_text(encoding=encoding, errors="ignore")
        except OSError:
            continue
    return ""


def read_notebook_code(path):
    try:
        notebook = json.loads(read_text_file(path))
    except json.JSONDecodeError:
        return ""

    cells = []
    for cell in notebook.get("cells", []):
        if cell.get("cell_type") != "code":
            continue
        source = cell.get("source", [])
        source = "".join(source) if isinstance(source, list) else source
        if source and source.rstrip():
            cells.append(source.rstrip())
    return "\n\n# %%\n\n".join(cells) + ("\n" if cells else "")


def extract_text(path):
    return (
        read_notebook_code(path)
        if path.suffix.lower() == ".ipynb"
        else read_text_file(path)
    )


def count_words(text):
    return len(re.findall(r"[A-Za-z0-9_]+", text))


def count_lines(text):
    return len(text.splitlines())


def path_penalty_or_bonus(rel_path):
    normalized = rel_path.lower().replace("\\", "/")
    score = sum(
        value
        for pattern, value in BONUS_PATH_PATTERNS
        if re.search(pattern, normalized)
    )
    score += sum(
        value
        for pattern, value in PENALTY_PATH_PATTERNS
        if re.search(pattern, normalized)
    )
    if normalized.endswith((".min.js", ".min.css", ".bundle.js")):
        score -= 100
    return score


def looks_generated(text):
    return any(marker in text[:6000].lower() for marker in GENERATED_MARKERS)


def looks_minified(text, rel_path):
    if rel_path.lower().endswith((".min.js", ".min.css", ".bundle.js")):
        return True
    lines = text.splitlines()
    if len(lines) < 8:
        return False
    long_lines = sum(len(line) > 450 for line in lines)
    average_length = sum(map(len, lines)) / len(lines)
    return (long_lines / len(lines) > 0.35 and average_length > 160) or (
        "\n" not in text[:5000] and len(text) > 5000
    )


def import_heavy_ratio(text):
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    if not lines:
        return 1.0
    pattern = re.compile(
        r"^(import\s|from\s.+\simport\s|#include\s|using\s+namespace|using\s+[A-Za-z0-9_.:]+\s*=|require\(|package\s+[A-Za-z0-9_.]+;)"
    )
    head = lines[:120]
    return sum(bool(pattern.search(line)) for line in head) / len(head)


def comment_ratio(text):
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    if not lines:
        return 1.0
    return sum(line.startswith(("#", "//", "/*", "*", "--")) for line in lines) / len(
        lines
    )


def repeated_line_ratio(text):
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    if not lines:
        return 1.0
    counts = Counter(lines)
    return sum(count for count in counts.values() if count > 1) / len(lines)


def syntax_score(text, ext):
    lower_text = text.lower()
    score = 0
    score += 4 * len(
        re.findall(
            r"\b(if|else|elif|switch|case|for|foreach|while|do|try|catch|except|finally|throw|return|yield|break|continue)\b",
            lower_text,
        )
    )
    score += 4 * len(
        re.findall(
            r"\b(def|function|func|class|interface|enum|struct|trait|impl|namespace|template|lambda|async|await)\b",
            lower_text,
        )
    )
    score += 3 * len(
        re.findall(
            r"\b(public|private|protected|extends|implements|override|virtual|static|const|final|new|super|self|this)\b",
            lower_text,
        )
    )
    score += 4 * len(
        re.findall(
            r"\b(select|insert|update|delete|join|where|group by|order by|create table|alter table|schema|transaction|commit|rollback)\b",
            lower_text,
        )
    )
    score += 3 * len(
        re.findall(
            r"\b(auth|token|jwt|bcrypt|hash|encrypt|decrypt|serialize|deserialize|validate|parser?|compile|solver|algorithm|graph|tree|heap|queue|stack)\b",
            lower_text,
        )
    )
    score += 3 * len(
        re.findall(
            r"\b(request|response|router|route|endpoint|server|client|http|https|api|fetch|axios|requests|socket)\b",
            lower_text,
        )
    )

    if ext == ".py":
        score += 8 * len(re.findall(r"(?m)^\s*def\s+[A-Za-z_][A-Za-z0-9_]*\s*\(", text))
        score += 8 * len(
            re.findall(r"(?m)^\s*class\s+[A-Za-z_][A-Za-z0-9_]*\s*[:(]", text)
        )
    elif ext in {".js", ".ts", ".jsx", ".tsx", ".vue", ".svelte"}:
        score += 7 * len(
            re.findall(r"\bfunction\s+[A-Za-z_$][A-Za-z0-9_$]*\s*\(", text)
        )
        score += 6 * len(
            re.findall(r"\b[A-Za-z_$][A-Za-z0-9_$]*\s*=\s*\([^)]*\)\s*=>", text)
        )
        score += 6 * len(re.findall(r"\bclass\s+[A-Za-z_$][A-Za-z0-9_$]*", text))
    elif ext in {
        ".java",
        ".kt",
        ".kts",
        ".c",
        ".cc",
        ".cpp",
        ".cxx",
        ".h",
        ".hh",
        ".hpp",
        ".cs",
        ".go",
        ".rs",
        ".swift",
        ".scala",
        ".php",
        ".rb",
        ".dart",
        ".m",
    }:
        score += 7 * len(
            re.findall(r"\b(class|interface|struct|enum|impl|trait|namespace)\b", text)
        )
        score += 5 * len(
            re.findall(
                r"\b[A-Za-z_][A-Za-z0-9_<>:\[\]]*\s+[A-Za-z_][A-Za-z0-9_]*\s*\([^;\n]*\)\s*\{",
                text,
            )
        )
    elif ext == ".sql":
        score += 12 * len(
            re.findall(
                r"\b(select|insert|update|delete|create|alter|drop|join|where|group by|order by)\b",
                lower_text,
            )
        )
    elif ext in {".sh", ".bash", ".zsh"}:
        score += 6 * len(
            re.findall(r"\b(if|then|fi|for|while|case|function)\b", lower_text)
        )

    score += min(sum(bool(line.strip()) for line in text.splitlines()), 400) // 6
    html_tags = len(re.findall(r"<[A-Za-z][^>]*>", text))
    css_properties = len(
        re.findall(
            r"\b(color|margin|padding|font|display|position|background|border)\s*:",
            lower_text,
        )
    )
    if ext in {".html", ".css", ".scss", ".sass"}:
        score -= int((html_tags + css_properties) * 0.6)
    elif html_tags > 250 and ext in {".jsx", ".tsx", ".vue", ".svelte"}:
        score -= int(html_tags * 0.12)

    score -= int(comment_ratio(text) * 40)
    score -= int(repeated_line_ratio(text) * 70)
    if import_heavy_ratio(text) > 0.82:
        score -= 35
    if looks_generated(text):
        score -= 60
    return score


def keep_file(path):
    extension = path.suffix.lower()
    if (
        path.name.lower() in {name.lower() for name in SKIP_FILES}
        or extension in BINARY_EXTS
        or extension not in CODE_EXTS
    ):
        return False
    try:
        return path.stat().st_size <= MAX_FILE_BYTES and not is_probably_binary(path)
    except OSError:
        return False


def candidate_score(rel_path, text):
    extension = Path(rel_path).suffix.lower()
    if extension not in CODE_EXTS:
        return 0, {"reason": "unsupported_extension"}
    if not text.strip():
        return 0, {"reason": "empty"}

    lines, words = count_lines(text), count_words(text)
    if lines < MIN_LINES_PER_FILE:
        return 0, {"reason": "too_short", "lines": lines, "words": words}
    if lines > MAX_LINES_PER_FILE:
        return 0, {"reason": "too_long", "lines": lines, "words": words}
    if looks_minified(text, rel_path):
        return 0, {"reason": "minified", "lines": lines, "words": words}

    score = (
        EXT_BASE_BONUS.get(extension, 40)
        + path_penalty_or_bonus(rel_path)
        + syntax_score(text, extension)
    )
    if extension in {".html", ".css", ".scss", ".sass"} and score < 60:
        score -= 40
    return max(0, score), {"lines": lines, "words": words, "ext": extension}


def collect_candidates(repo_root, base_path):
    candidates, seen_hashes = [], set()
    for root, directories, filenames in os.walk(base_path):
        directories[:] = [
            directory for directory in directories if directory not in SKIP_DIRS
        ]
        for filename in filenames:
            path = Path(root) / filename
            if not keep_file(path):
                continue
            text = extract_text(path)
            if not text.strip():
                continue
            digest = hashlib.sha1(
                text[:250000].encode("utf-8", errors="ignore")
            ).hexdigest()
            if digest in seen_hashes:
                continue
            seen_hashes.add(digest)

            relative_path = str(path.relative_to(repo_root)).replace("\\", "/")
            score, metadata = candidate_score(relative_path, text)
            if score > 0:
                candidates.append(
                    {
                        "file_rel": relative_path,
                        "text": text,
                        "score": score,
                        **metadata,
                    }
                )

    candidates.sort(
        key=lambda item: (item["score"], item["lines"], item["words"]), reverse=True
    )
    return candidates[:MAX_CANDIDATE_FILES_PER_REPO]


def choose_files(candidates):
    selected, total_lines, total_words = [], 0, 0
    for candidate in candidates:
        if len(selected) >= MAX_SELECTED_FILES_PER_REPO:
            break
        selected.append(candidate)
        total_lines += candidate["lines"]
        total_words += candidate["words"]
        if total_lines >= TARGET_MIN_LINES or total_words >= TARGET_MIN_WORDS:
            break

    reached = total_lines >= TARGET_MIN_LINES or total_words >= TARGET_MIN_WORDS
    return (
        ([] if not reached and not WRITE_SMALL_OUTPUTS else selected),
        total_lines,
        total_words,
        reached,
    )


def write_final_txt(path, selected):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", errors="ignore") as handle:
        for index, candidate in enumerate(selected):
            if index:
                handle.write(f"\n{MARK_LINE}\n\n")
            handle.write(f"{candidate['file_rel'].strip()}\n")
            handle.write(
                candidate["text"]
                if candidate["text"].endswith("\n")
                else f"{candidate['text']}\n"
            )


def write_candidate_report(
    path, url, candidates, selected, total_lines, total_words, reached
):
    path.parent.mkdir(parents=True, exist_ok=True)
    selected_paths = {candidate["file_rel"] for candidate in selected}
    with path.open("w", encoding="utf-8", errors="ignore") as handle:
        handle.write(
            f"URL: {url}\nCandidates found: {len(candidates)}\nSelected files: {len(selected)}\n"
        )
        handle.write(
            f"Selected total lines: {total_lines}\nSelected total words: {total_words}\n"
        )
        handle.write(
            f"Threshold reached ({TARGET_MIN_LINES} lines OR {TARGET_MIN_WORDS} words): {reached}\n\n"
        )
        handle.write("RANK | SELECTED | SCORE | LINES | WORDS | EXT | FILE\n")
        handle.write("-" * 120 + "\n")
        for rank, candidate in enumerate(candidates, 1):
            selected_mark = "YES" if candidate["file_rel"] in selected_paths else "NO"
            handle.write(
                f"{rank:>4} | {selected_mark:^8} | {candidate['score']:>5} | {candidate['lines']:>5} | {candidate['words']:>5} | {candidate['ext']:<6} | {candidate['file_rel']}\n"
            )


def process_one_repo(url, temporary_parent):
    info = gh_parse(url)
    if not info:
        return {
            "status": "bad_url",
            "url": url,
            "message": "Could not parse GitHub URL.",
        }

    stem = output_stem(info, url)
    repo_dir = temporary_parent / stem
    cloned, clone_log = git_clone_repo(info["owner"], info["repo"], repo_dir)
    if not cloned:
        return {
            "status": "clone_failed",
            "url": url,
            "owner": info["owner"],
            "repo": info["repo"],
            "stem": stem,
            "message": clone_log,
        }

    base_path = repo_dir
    if info["kind"] in {"tree", "blob"} and info["subpath"]:
        candidate_path = repo_dir / info["subpath"]
        if candidate_path.exists():
            base_path = (
                candidate_path if candidate_path.is_dir() else candidate_path.parent
            )

    candidates = collect_candidates(repo_dir, base_path)
    if not candidates:
        return {
            "status": "no_candidates",
            "url": url,
            "owner": info["owner"],
            "repo": info["repo"],
            "stem": stem,
            "message": "No qualifying code files found after trimming/scoring.",
        }

    selected, total_lines, total_words, reached = choose_files(candidates)
    if not selected:
        return {
            "status": "below_threshold",
            "url": url,
            "owner": info["owner"],
            "repo": info["repo"],
            "stem": stem,
            "candidate_count": len(candidates),
            "message": "Output writing was disabled for a repository below the target threshold.",
        }

    output_path = FINAL_ROOT / f"{stem}.txt"
    report_path = REPORT_ROOT / f"{stem}.txt"
    write_final_txt(output_path, selected)
    write_candidate_report(
        report_path, url, candidates, selected, total_lines, total_words, reached
    )
    return {
        "status": "written" if reached else "written_small",
        "url": url,
        "owner": info["owner"],
        "repo": info["repo"],
        "stem": stem,
        "candidate_count": len(candidates),
        "selected_count": len(selected),
        "selected_lines": total_lines,
        "selected_words": total_words,
        "threshold_reached": reached,
        "output_path": str(output_path),
        "report_path": str(report_path),
        "selected": selected,
    }


def write_failure_log(failures):
    FAILURE_LOG.parent.mkdir(parents=True, exist_ok=True)
    with FAILURE_LOG.open("w", encoding="utf-8", errors="ignore") as handle:
        if not failures:
            handle.write("(none)\n")
            return
        for failure in failures:
            handle.write(
                f"{'=' * 100}\nURL: {failure.get('url', '')}\nSTATUS: {failure.get('status', '')}\nMESSAGE:\n{failure.get('message', '')}\n\n"
            )


def write_summary(rows):
    OUT_ROOT.mkdir(parents=True, exist_ok=True)
    fields = [
        "url",
        "owner",
        "repo",
        "status",
        "candidate_count",
        "selected_count",
        "selected_lines",
        "selected_words",
        "threshold_reached",
        "output_path",
        "report_path",
    ]
    with SUMMARY_CSV.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        for row in rows:
            writer.writerow({field: row.get(field, "") for field in fields})


def write_file_level_summary(rows):
    fields = [
        "url",
        "owner",
        "repo",
        "status",
        "selected_rank",
        "file_rel",
        "ext",
        "score",
        "lines",
        "words",
    ]
    with FILE_LEVEL_CSV.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        for row in rows:
            for rank, item in enumerate(row.get("selected", []), 1):
                writer.writerow(
                    {
                        "url": row.get("url", ""),
                        "owner": row.get("owner", ""),
                        "repo": row.get("repo", ""),
                        "status": row.get("status", ""),
                        "selected_rank": rank,
                        "file_rel": item.get("file_rel", ""),
                        "ext": item.get("ext", ""),
                        "score": item.get("score", 0),
                        "lines": item.get("lines", 0),
                        "words": item.get("words", 0),
                    }
                )


def existing_final_txt_count():
    FINAL_ROOT.mkdir(parents=True, exist_ok=True)
    return len(list(FINAL_ROOT.glob("*.txt")))


def load_existing_summary_rows():
    if not SUMMARY_CSV.exists():
        return []
    try:
        with SUMMARY_CSV.open(
            "r", encoding="utf-8", errors="ignore", newline=""
        ) as handle:
            return list(csv.DictReader(handle))
    except (OSError, csv.Error) as exc:
        print(f"Warning: could not load previous summary rows: {exc}")
        return []


def previously_written_urls(rows):
    return {
        normalize_url(row.get("url", ""))
        for row in rows
        if row.get("status", "").strip() in {"written", "written_small"}
    }


def append_summary_rows(existing_rows, new_rows):
    seen, merged = set(), []
    for row in [*existing_rows, *new_rows]:
        key = (row.get("url", ""), row.get("output_path", ""), row.get("status", ""))
        if key not in seen:
            seen.add(key)
            merged.append(row)
    return merged


def main():
    urls = load_urls()
    for directory in (OUT_ROOT, FINAL_ROOT, REPORT_ROOT):
        directory.mkdir(parents=True, exist_ok=True)

    existing_rows = load_existing_summary_rows()
    completed_urls = previously_written_urls(existing_rows)
    existing_count = existing_final_txt_count()

    print(f"Loaded {len(urls)} GitHub URLs")
    print(f"SOURCE_MODE = {SOURCE_MODE}")
    print(f"OUT_ROOT = {OUT_ROOT}")
    print(f"Existing final TXT files = {existing_count}")
    print(f"Target final TXT files = {TARGET_FINAL_FILES}")

    if existing_count >= TARGET_FINAL_FILES:
        print("Target already met. No new processing needed.")
        return

    urls_to_process = [url for url in urls if normalize_url(url) not in completed_urls]
    new_rows, failures, current_count = [], [], existing_count

    with tempfile.TemporaryDirectory() as temporary_directory:
        temporary_parent = Path(temporary_directory)
        for index, url in enumerate(urls_to_process, 1):
            if current_count >= TARGET_FINAL_FILES:
                print(f"Reached target of {TARGET_FINAL_FILES} final TXT files.")
                break

            print(f"[{index}/{len(urls_to_process)}] {url}")
            result = process_one_repo(url, temporary_parent)
            new_rows.append(result)
            status = result.get("status", "unknown")

            if status in {
                "clone_failed",
                "bad_url",
                "no_candidates",
                "below_threshold",
            }:
                failures.append(result)
                print(f"  -> {status}")
            else:
                current_count += 1
                print(
                    f"  -> {status} | selected_files={result.get('selected_count', 0)} | lines={result.get('selected_lines', 0)} | words={result.get('selected_words', 0)} | final_txt_total={current_count}"
                )

    write_summary(append_summary_rows(existing_rows, new_rows))
    # Historical selected-file details are not recoverable from summary.csv.
    write_file_level_summary(new_rows)
    write_failure_log(failures)

    print("\nDONE")
    for status, count in sorted(
        Counter(row.get("status", "unknown") for row in new_rows).items()
    ):
        print(f"  {status}: {count}")
    print(f"\nFinal TXT files: {existing_final_txt_count()}")
    print(f"Final outputs: {FINAL_ROOT}")
    print(f"Candidate reports: {REPORT_ROOT}")
    print(f"Summary CSV: {SUMMARY_CSV}")
    print(f"File-level CSV: {FILE_LEVEL_CSV}")
    print(f"Failure log: {FAILURE_LOG}")


if __name__ == "__main__":
    main()

# Trimming Mixed-Language GitHub Repositories for AI Detection

## Repository extraction workflow

The notebook processes mixed-language repositories from the provided GitHub URLs. It can also read URLs from `Coding Projects URLs.pdf` in Drive.

For each repository, it:

- clones the repository;
- scores files across multiple languages and file types rather than assuming JavaScript web projects;
- retains files most likely to contain student-authored logic;
- writes one trimmed `.txt` file per repository for AI-detection input; and
- generates candidate reports and CSV summaries.

Each AI-detection text file contains the relative path followed by the file's raw code for every retained file. Files are separated by:

```text
############
```

In [ ]:
# Colab setup
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

!pip install --quiet pypdf

import csv
import hashlib
import json
import os
import re
import shutil
import subprocess
import tempfile
from collections import Counter
from pathlib import Path
from urllib.parse import urlparse

from pypdf import PdfReader

SOURCE_MODE = "hardcoded"  # "hardcoded" or "pdf"
DATA_ROOT = Path(os.environ.get("PROJECT_DATA_ROOT", "./project_data"))
PDF_WITH_URLS = Path(os.environ.get("PROJECT_URL_PDF", DATA_ROOT / "repository_urls.pdf"))

OUT_ROOT = DATA_ROOT
FINAL_ROOT = OUT_ROOT / "trimmed_for_detection"
REPORT_ROOT = OUT_ROOT / "candidate_reports"
SUMMARY_CSV = OUT_ROOT / "summary.csv"
FILE_LEVEL_CSV = OUT_ROOT / "selected_files.csv"
FAILURE_LOG = OUT_ROOT / "failures.txt"

# Stop selection after either corpus-size threshold is met.
TARGET_MIN_LINES = 800
TARGET_MIN_WORDS = 3000
WRITE_SMALL_OUTPUTS = True

MIN_LINES_PER_FILE = 12
MAX_LINES_PER_FILE = 4000
MAX_FILE_BYTES = 1_200_000
MAX_CANDIDATE_FILES_PER_REPO = 60
MAX_SELECTED_FILES_PER_REPO = 25
MARK_LINE = "############"

HARDCODED_URLS = """
https://github.com/MrAnas/ICS324_GROUP_2
https://github.com/KonstantinFilin/ksutils
https://github.com/djbamba/Wildcat-Chat
https://github.com/HassanAlgoz/chefs-website
https://github.com/norahalkhunifer/web
https://github.com/lameessu/Bohemia-website
https://github.com/raisking/ksu_webproject
https://github.com/ksuwannop/ksuwannop.github.io
https://github.com/MAD9013/responsive-website-project1-kaus0025
https://github.com/baileythompson/KSU-IT-Research_Website
https://github.com/ksudhanshu348/ksudhanshu348.github.io
https://github.com/hannahnoelle797/IntroToWebDevSite
https://github.com/hannahnoelle797/AdvWebDevSite
https://github.com/rishamitha/ksu-cafeteria
https://github.com/kaushiki7/kaushiki7.github.io
https://github.com/Narwher/la-hair-atl
https://github.com/pkommineni94/IT6203
https://github.com/surafel911/pizza-shop-webservice
https://github.com/jsweitz1/KSU-Student
https://github.com/othmanKisha/In-Door-Face-Mask-Inspector
https://github.com/ksulem/ksulem.github.io
https://github.com/MujtabaMohsin/Student-Exam-Preparation
https://github.com/Reemaad/KSU-Templates
https://github.com/Kaushikodey/Kaushikodey.github.io
https://github.com/ksuni338/ksuni338.github.io
https://github.com/baileyknez/CapstoneKSU2021
https://github.com/iamanilantony/KSU
https://github.com/ericbutler555/carit
https://github.com/traviskaufman/TravisKaufman_FlashMP3Player
https://github.com/danielgur/ksu-buzz-out
https://github.com/ibalde/colesAppWebPage
https://github.com/jsturzio/websiteacm
https://github.com/lemonad/ksu-online
https://github.com/ksusanth/ksusanth.github.io
https://github.com/santoslab/ksu-cis-501-web
https://github.com/ksuchow99/ksuchow99.github.io
https://github.com/willbreitkreutz/ksu_geog_alumni
https://github.com/hyttjo/KaupunkiUutiset
https://github.com/MaximTikhonov/WebDesignKsu
https://github.com/KeralaStartupMission/ksum_fellows
https://github.com/ksw2342/kau_webstudio_12
https://github.com/dp3/ksudrupal7
https://github.com/zeuxisoo/web-kauppaa
https://github.com/Front-End-Technology/KaushikPizzaWeb
https://github.com/Supan13/kaur-supan-webdev
https://github.com/juhapekkaeloranta/kaupunkifillarit-web
https://github.com/ajaykauthale1/kauthale-ajay-webdev
https://github.com/kaushikveluru/veluru-kaushik-webdev
https://github.com/Supan13/kaur-supan13-webdev
https://github.com/kevin-su/ksu-reactjs
""".split()

GITHUB_URL_RE = re.compile(r'https?://[^\s)\]"\'>]+', re.I)
CODE_EXTS = set(".py .js .ts .jsx .tsx .java .kt .kts .c .cc .cpp .cxx .h .hh .hpp .cs .go .rs .rb .php .swift .scala .sql .html .css .scss .sass .m .r .jl .sh .bash .zsh .vue .svelte .dart .lua .pl .ipynb".split())
BINARY_EXTS = set(".png .jpg .jpeg .gif .bmp .webp .ico .pdf .zip .rar .7z .tar .gz .mp3 .wav .mp4 .avi .mov .mkv .class .jar .exe .dll .so .o .a .pyc .pyo .woff .woff2 .ttf .eot .map".split())
SKIP_DIRS = set(".git .github .idea .vscode __pycache__ node_modules vendor dist build out coverage .next .nuxt .svelte-kit .cache .gradle .mypy_cache .pytest_cache .tox venv .venv env Pods target bin obj static assets images img fonts".split())
SKIP_FILES = set("package-lock.json yarn.lock pnpm-lock.yaml composer.lock poetry.lock Cargo.lock Gemfile.lock Pipfile.lock pubspec.lock LICENSE LICENSE.md COPYING favicon.ico".split())
GENERATED_MARKERS = ["do not edit", "this file was generated", "generated by", "autogenerated", "auto-generated", "automatically generated", "create-react-app", "vite", "webpack bootstrap", "@generated"]

EXT_BASE_BONUS = dict(zip(
    ".py .java .kt .kts .c .cc .cpp .cxx .h .hh .hpp .cs .go .rs .rb .php .swift .scala .sql .js .ts .jsx .tsx .vue .svelte .dart .m .r .jl .sh .bash .zsh .html .css .scss .sass .lua .pl .ipynb".split(),
    [90, 88, 88, 70, 85, 85, 85, 85, 55, 55, 55, 90, 90, 92, 80, 80, 84, 82, 82, 78, 82, 64, 68, 70, 70, 78, 74, 74, 74, 56, 56, 56, 28, 14, 18, 18, 72, 70, 78],
))
BONUS_PATH_PATTERNS = [(r"(^|/)(src|app|server|backend|api|core|lib|internal|pkg|cmd|domain|logic|engine)(/|$)", 35), (r"(^|/)(controllers?|services?|handlers?|routes?|routers?|middleware|guards?|auth|security)(/|$)", 45), (r"(^|/)(models?|entities?|dto|schemas?|database|db|repositories?|dao|migrations?)(/|$)", 35), (r"(^|/)(parser|compiler|solver|algorithm|algorithms|graph|tree|heap|search|sort)(/|$)", 35), (r"(^|/)(utils?|helpers?|shared|common)(/|$)", 18)]
PENALTY_PATH_PATTERNS = [(r"(^|/)(test|tests|testing|__tests__|spec|specs|fixtures|mocks|mock|sample|samples|example|examples)(/|$)", -35), (r"(^|/)(public|docs?|documentation|assets?|images?|img|fonts?|styles?|css)(/|$)", -28), (r"(^|/)(vendor|third_party|third-party|external|generated|dist|build|coverage)(/|$)", -45)]

def run(cmd, cwd=None):
    p = subprocess.run(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    return p.returncode, p.stdout

def normalize_url(url): return (url or "").strip().rstrip(".,;)")
def ordered_unique(items): return list(dict.fromkeys(items))
def is_github_url(url):
    try: return "github.com" in urlparse(url).netloc.lower()
    except Exception: return False

def read_urls_from_pdf(path):
    if not path.exists(): raise FileNotFoundError(f"PDF not found: {path}")
    return ordered_unique(normalize_url(u) for p in PdfReader(str(path)).pages for u in GITHUB_URL_RE.findall(p.extract_text() or "") if is_github_url(normalize_url(u)))

def load_urls():
    urls = read_urls_from_pdf(PDF_WITH_URLS) if SOURCE_MODE.lower() == "pdf" else map(normalize_url, HARDCODED_URLS)
    urls = [u for u in ordered_unique(urls) if is_github_url(u)]
    if not urls: raise ValueError("No GitHub URLs were loaded.")
    return urls

def gh_parse(url):
    m = re.match(r"https?://github\.com/([^/]+)/([^/]+)(?:/(.*))?$", url.strip())
    if not m: return None
    owner, repo, tail = m.group(1), m.group(2).replace(".git", ""), (m.group(3) or "").strip("/")
    info = {"owner": owner, "repo": repo, "kind": "repo", "subpath": None, "raw_url": url}
    if tail.startswith(("tree/", "blob/")):
        parts = tail.split("/", 2); info["kind"] = parts[0]; info["subpath"] = parts[2] if len(parts) > 2 else None
    return info

def safe_slug(text): return re.sub(r"[^A-Za-z0-9._-]+", "_", text).strip("._") or "item"
def output_stem(info, url): return safe_slug(f"{info['owner']}__{info['repo']}__{hashlib.sha1(url.encode()).hexdigest()[:10]}")

def git_clone_repo(owner, repo, dest):
    dest.parent.mkdir(parents=True, exist_ok=True); shutil.rmtree(dest, ignore_errors=True)
    first = ["git", "clone", "--depth", "1", "--single-branch", f"https://github.com/{owner}/{repo}.git", str(dest)]
    rc, out = run(first)
    if rc == 0: return True, out
    shutil.rmtree(dest, ignore_errors=True)
    rc, retry = run(["git", "clone", "--depth", "1", f"https://github.com/{owner}/{repo}.git", str(dest)])
    return rc == 0, out + "\n\nRETRY:\n" + retry

def is_probably_binary(path):
    try: chunk = path.read_bytes()[:8192]
    except Exception: return True
    return b"\0" in chunk or (bool(chunk) and sum(b < 9 or 13 < b < 32 for b in chunk) / len(chunk) > .20)

def read_text_file(path):
    for encoding in ("utf-8", "utf-8-sig", "latin-1", "cp1252"):
        try: return path.read_text(encoding=encoding, errors="ignore")
        except Exception: pass
    return ""

def extract_text(path):
    if path.suffix.lower() != ".ipynb": return read_text_file(path)
    try: cells = json.loads(read_text_file(path)).get("cells", [])
    except Exception: return ""
    code = []
    for cell in cells:
        if cell.get("cell_type") == "code":
            source = cell.get("source", []); source = "".join(source) if isinstance(source, list) else source
            if source and source.rstrip(): code.append(source.rstrip())
    return "\n\n# %%\n\n".join(code) + ("\n" if code else "")

def count_words(text): return len(re.findall(r"[A-Za-z0-9_]+", text))
def count_lines(text): return len(text.splitlines())

def path_score(rel):
    rel = rel.lower().replace("\\", "/"); score = sum(v for p, v in BONUS_PATH_PATTERNS + PENALTY_PATH_PATTERNS if re.search(p, rel))
    return score - 100 if rel.endswith((".min.js", ".min.css", ".bundle.js")) else score

def looks_generated(text): return any(m in text[:6000].lower() for m in GENERATED_MARKERS)
def looks_minified(text, rel):
    lines = text.splitlines()
    return rel.lower().endswith((".min.js", ".min.css", ".bundle.js")) or (len(lines) >= 8 and sum(len(x) > 450 for x in lines) / len(lines) > .35 and sum(map(len, lines)) / len(lines) > 160) or ("\n" not in text[:5000] and len(text) > 5000)

def comment_ratio(text):
    lines = [x.strip() for x in text.splitlines() if x.strip()]
    return sum(x.startswith(("#", "//", "/*", "*", "--")) for x in lines) / len(lines) if lines else 1

def repeated_line_ratio(text):
    lines = [x.strip() for x in text.splitlines() if x.strip()]
    return sum(n for n in Counter(lines).values() if n > 1) / len(lines) if lines else 1

def candidate_score(rel, text):
    ext, lines, words = Path(rel).suffix.lower(), count_lines(text), count_words(text)
    if ext not in CODE_EXTS or not text.strip() or lines < MIN_LINES_PER_FILE or lines > MAX_LINES_PER_FILE or looks_minified(text, rel): return 0, {}
    keywords = r"\b(if|else|for|while|try|catch|return|def|function|class|interface|struct|async|await|select|insert|update|delete|join|auth|token|parser|algorithm|request|response|router|api)\b"
    score = EXT_BASE_BONUS.get(ext, 40) + path_score(rel) + 4 * len(re.findall(keywords, text.lower())) + min(lines, 400) // 6
    score -= int(comment_ratio(text) * 40) + int(repeated_line_ratio(text) * 70) + (60 if looks_generated(text) else 0)
    if ext in {".html", ".css", ".scss", ".sass"} and score < 60: score -= 40
    return max(0, score), {"lines": lines, "words": words, "ext": ext}

def keep_file(path):
    ext = path.suffix.lower()
    try: size = path.stat().st_size
    except Exception: return False
    return path.name.lower() not in {x.lower() for x in SKIP_FILES} and ext in CODE_EXTS and ext not in BINARY_EXTS and size <= MAX_FILE_BYTES and not is_probably_binary(path)

def collect_candidates(repo_root, base_path):
    candidates, hashes = [], set()
    for root, dirs, files in os.walk(base_path):
        dirs[:] = [d for d in dirs if d not in SKIP_DIRS]
        for name in files:
            path = Path(root) / name
            if not keep_file(path): continue
            text = extract_text(path); digest = hashlib.sha1(text[:250000].encode("utf-8", errors="ignore")).hexdigest()
            if not text.strip() or digest in hashes: continue
            hashes.add(digest); rel = str(path.relative_to(repo_root)).replace("\\", "/"); score, meta = candidate_score(rel, text)
            if score: candidates.append({"file_rel": rel, "text": text, "score": score, **meta})
    return sorted(candidates, key=lambda x: (x["score"], x["lines"], x["words"]), reverse=True)[:MAX_CANDIDATE_FILES_PER_REPO]

def choose_files(candidates):
    selected, lines, words = [], 0, 0
    for item in candidates:
        if len(selected) >= MAX_SELECTED_FILES_PER_REPO: break
        selected.append(item); lines += item["lines"]; words += item["words"]
        if lines >= TARGET_MIN_LINES or words >= TARGET_MIN_WORDS: break
    reached = lines >= TARGET_MIN_LINES or words >= TARGET_MIN_WORDS
    return ([] if not reached and not WRITE_SMALL_OUTPUTS else selected), lines, words, reached

def write_final_txt(path, selected):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", errors="ignore") as f:
        for i, item in enumerate(selected):
            if i: f.write(f"\n{MARK_LINE}\n\n")
            f.write(item["file_rel"] + "\n" + item["text"].rstrip() + "\n")

def process_one_repo(url, tmp_parent):
    info = gh_parse(url)
    if not info: return {"status": "bad_url", "url": url, "message": "Could not parse GitHub URL."}
    stem, repo_dir = output_stem(info, url), tmp_parent / output_stem(info, url)
    ok, log = git_clone_repo(info["owner"], info["repo"], repo_dir)
    if not ok: return {"status": "clone_failed", "url": url, **info, "stem": stem, "message": log}
    base = repo_dir / info["subpath"] if info["subpath"] and (repo_dir / info["subpath"]).exists() else repo_dir
    if base.is_file(): base = base.parent
    candidates = collect_candidates(repo_dir, base)
    if not candidates: return {"status": "no_candidates", "url": url, **info, "stem": stem, "message": "No qualifying code files found after trimming/scoring."}
    selected, lines, words, reached = choose_files(candidates)
    if not selected: return {"status": "below_threshold", "url": url, **info, "stem": stem, "candidate_count": len(candidates), "message": "Output writing was disabled for this small repository."}
    out, report = FINAL_ROOT / f"{stem}.txt", REPORT_ROOT / f"{stem}.txt"
    write_final_txt(out, selected)
    return {"status": "written" if reached else "written_small", "url": url, **info, "stem": stem, "candidate_count": len(candidates), "selected_count": len(selected), "selected_lines": lines, "selected_words": words, "threshold_reached": reached, "output_path": str(out), "report_path": str(report), "selected": selected}

def write_summary(rows):
    OUT_ROOT.mkdir(parents=True, exist_ok=True)
    fields = ["url", "owner", "repo", "status", "candidate_count", "selected_count", "selected_lines", "selected_words", "threshold_reached", "output_path", "report_path"]
    with SUMMARY_CSV.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fields); writer.writeheader(); writer.writerows({k: row.get(k, "") for k in fields} for row in rows)
    with FILE_LEVEL_CSV.open("w", newline="", encoding="utf-8") as f:
        fields = ["url", "owner", "repo", "status", "selected_rank", "file_rel", "ext", "score", "lines", "words"]
        writer = csv.DictWriter(f, fieldnames=fields); writer.writeheader()
        for row in rows:
            for rank, item in enumerate(row.get("selected", []), 1): writer.writerow({"url": row.get("url", ""), "owner": row.get("owner", ""), "repo": row.get("repo", ""), "status": row.get("status", ""), "selected_rank": rank, **{k: item.get(k, "") for k in fields[5:]}})

def main():
    urls, rows = load_urls(), []
    FINAL_ROOT.mkdir(parents=True, exist_ok=True); REPORT_ROOT.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory() as tmp:
        for index, url in enumerate(urls, 1):
            print(f"[{index}/{len(urls)}] {url}")
            result = process_one_repo(url, Path(tmp)); rows.append(result); print(f"  -> {result['status']}")
    write_summary(rows)
    failures = [r for r in rows if r["status"] in {"clone_failed", "bad_url", "no_candidates", "below_threshold"}]
    FAILURE_LOG.write_text("\n\n".join(f"URL: {r.get('url', '')}\nSTATUS: {r['status']}\nMESSAGE:\n{r.get('message', '')}" for r in failures) or "(none)\n", encoding="utf-8")
    print("Status counts:", dict(Counter(r["status"] for r in rows)))

main()